# Introduction to HuggingFace for Molecular LLMs

---

## Learning Outcomes

By the end of this session, participants will be able to:
- Load datasets from HuggingFace using the `datasets` package
- Process the datasets that have been loaded
- Load and use tokenisers from HuggingFace

## credits

This notebooks is adapted from [Meg Hedge's ECCB2026 tutorial](https://github.com/meghegde/genomic-llms-in-practice/tree/main),
with idea and inspirations from [Raphael Mourad's genomic LLM finetuning on the Galaxy Training Network](https://training.galaxyproject.org/training-material/topics/statistics/tutorials/genomic-llm-finetuning/tutorial.html#solution-16) and [Hari Ramadasan's ELIXIR ML/AI 2026 workshop materials](https://github.com/hramdasan/elixir_summerschool_26/tree/main)

## Install and import required packages

> the installation should not be necessary on google colab

Uncomment and run the cell below if needed.

In [ ]:
#!pip install transformers datasets pandas transformers

*N.B.: If you perform installs, restart the runtime between the installs and imports*

In [ ]:
import pandas as pd
import torch
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoConfig, AutoModel

# Loading a dataset

HuggingFace hosts a large number of datasets that can be used to train, explore, and evaluate our models.

We are going to play with [InstaDeepAI/nucleotide_transformer_downstream_tasks_revised](https://huggingface.co/datasets/InstaDeepAI/nucleotide_transformer_downstream_tasks_revised) which contains train and test data for 18 classification tasks on DNA data.

You can read more on the data and tasks by clicking on the link above.


In [ ]:
%%time
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")
print(dataset)

This gives us a Huggingface DatasetDict, containing two Huggingface Dataset objects, one for the train and one for the test split.

In [ ]:
train_dataset = dataset["train"]
print(train_dataset)

A Huggingface Dataset can be converted into a Pandas DataFrame.

In [ ]:
train_df = train_dataset.to_pandas()
train_df.head()

You can then do all the usual Pandas manipulations, which can be helpful for inspecting the dataset.

In [ ]:
train_df[train_df.task=="H2AFZ"]

In [ ]:
train_df[train_df.task=="H2AFZ"].label.value_counts()

In [ ]:
train_df.task.value_counts()

The Huggingface Dataset has its own functions that let you manipulate it directly.


*  `train_test_split` : can be used to further divide the dataset (into a training and a validation set for example):

In [ ]:
# produces another DatasetDict, containing two Datasets - one "train" and the other "test"
split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset



* `select`: to extract specific rows by index

In [ ]:
small_dataset = train_dataset.select(range(50))
print(small_dataset)
print(len(small_dataset))

* `filter`: to extract rows based on some property





In [ ]:
filtered_train_dataset = train_dataset.filter(lambda x: x["task"]=="H2AFZ")
print(filtered_train_dataset)
filtered_train_dataset.to_pandas().task.value_counts()

---

Actually, we are going to generate right now a version of the dataset focused on the H2AFZ task (which is about  predicting whether it contains a binding site for the H2AFZ histone variant)

In [ ]:
dataset = dataset.filter(lambda x: x["task"]=="H2AFZ")
dataset

# Tokeniser

* The tokeniser is used to prepare the data for model input.
* It breaks a sequence into smaller chunks or "tokens".
* A **tokenizer is specific to a model**

Related Huggingface pages:
* https://huggingface.co/docs/transformers/en/main_classes/tokenizer
* https://huggingface.co/learn/llm-course/chapter2/4

## Loading a tokenizer

* This tokeniser splits sequences into 6-mers - chunks of 6 nucleotides - when possible, otherwise tokenising each nucleotide separately.
* This tokenizer has a vocabulary size of 4105. The inputs of the model are then of the form:
```
<CLS> <ACGTGT> <ACGTGC> <ACGGAC> <GACTAG> <TCAGCA>
```
* The tokenised sequences have a maximum length of 1,000.

Related Huggingface page: https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species


In [ ]:
%%time
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In general the code above is fine, but just to make things more explicit we are going to repeat the above with some parameters:

In [ ]:
tokeniser = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=1000,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)



* `model_max_length=1000`: Sets the maximum length of the tokenized sequences. Sequences longer than this will be truncated, and shorter ones will be padded.
* `padding_side="right"`: Specifies that padding should be added to the right side of the sequences. This ensures that all sequences in a batch have the same length.
* `use_fast=True`: Enables the use of the fast tokenizer implementation, which is optimized for speed and is suitable for most use cases.
* `trust_remote_code=True`: Allows the tokenizer to execute custom code from the model repository, which may be necessary for some models that require specific preprocessing steps.

*from [galaxy's genomic llm finetuning](https://training.galaxyproject.org/training-material/topics/statistics/tutorials/genomic-llm-finetuning/tutorial.html#solution-16)*

## Tokenize an example sequence

Let's get an example sequence

In [ ]:
sequence = "ATATTTTTCGGTGTTTTTTTAAAATCCAGATT"
print(sequence)
print(len(sequence))

By passing the sequence into the tokeniser, we get the input ids and attention mask.

In [ ]:
tok = tokeniser(sequence)
print(tok)

In [ ]:
token_ids = tok["input_ids"]
print(token_ids)

In [ ]:
attention_mask = tok["attention_mask"]
print(attention_mask)

By using the `tokenize` function, we get the tokens themselves.

In [ ]:
tokens = tokeniser.tokenize(sequence)
print(tokens)

To decode the sequence, we can put the input ids into the `decode` function.

In [ ]:
decoded = tokeniser.decode(token_ids)
print(decoded)

In practice, models often require their input to be of a fixed length.
To do achieve this we **pad** the sequences:

In [ ]:
tok = tokeniser(sequence, padding = 'max_length')
print(tok['input_ids'])
print(tok['attention_mask'])

In [ ]:
decoded = tokeniser.decode(tok['input_ids'])
print(decoded)

You can learn more about the tokenizer object from itself:

In [ ]:
print( "tokenizer vocabulary size", tokeniser.vocab_size )
print( "tokenizer special tokens", tokeniser.special_tokens_map )

### Exercise: Tokenise the example sequence for DNABERT-2
Huggingface page for [zhihan1996/DNABERT-2-117M](https://huggingface.co/zhihan1996/DNABERT-2-117M)

In [ ]:
# Get the model name
MODEL_NAME =

In [ ]:
# Load the tokeniser
# Use a maximum length of 512bp
# Pad on the left side only
tokeniser =

In [ ]:
# Get the tokenised output

In [ ]:
# Get the input_ids

In [ ]:
# Decode the input_ids

#### Answer

In [ ]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"

In [ ]:
%%time
# Load the tokeniser
# Use a maximum length of 512bp
# Pad on the left side only
tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="left",
    use_fast=True,
    trust_remote_code=True,
)

In [ ]:
# Get the tokenised output
tokens = tokeniser.tokenize(sequence)
print(tokens)

In [ ]:
# Get the input_ids
input_ids = tokeniser(sequence)["input_ids"]
print(input_ids)

In [ ]:
# Decode the input_ids
decoded = tokeniser.decode(input_ids)
print(decoded)

## Tokenise several sequences

We will come back to [InstaDeepAI/nucleotide-transformer-v2-50m-multi-species](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species)

In [ ]:
MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"

tokeniser = transformers.AutoTokenizer.from_pretrained(
    MODEL_NAME,
    model_max_length=512,
    padding_side="right",
    use_fast=True,
    trust_remote_code=True,
)

### Tokenize a list of sequences

Let's select the first five sequences in the dataset as our batch.

In [ ]:
batch = train_dataset["sequence"][:5]
print(len(batch))
print(batch)

The tokeniser can natively handle batches.

In [ ]:
encoded_batch = tokeniser(batch)
print(encoded_batch)

The `tokenize` function does not have a native batch mode; instead, it process one string at a time.

In [ ]:
tokens = tokeniser.tokenize(batch)
print(tokens)

Instead:

In [ ]:
tokens = [tokeniser.tokenize(seq) for seq in batch]
print(tokens)

### Tokenise a whole dataset

We could extract all the sequences in our Dataset a to a list and tokenize them as we have just done, but that is not the most efficient way to do things! When working with large datasets or tokenising multiple columns, it is preferable to use the `.map` function.

This is a built-in function of the Huggingface dataset object, and efficiently handles batches and multiprocessing. This approach avoids loading all sequences into memory at once.

However, this is not the most efficient way to do things! When working with large datasets or tokenising multiple columns, it is preferable to use the `.map` function.

This is a built-in function of the Huggingface dataset object, and efficiently handles batches and multiprocessing. This approach avoids loading all sequences into memory at once.

In [ ]:
train_dataset = dataset['train']

We first define a tokenise function.

In [ ]:
def tokenise_batch(batch):
    return tokeniser(
        batch["sequence"], # Select only the sequence column
        padding=True,
        truncation=True,
        max_length=512,
    )

We then apply this to the dataset using `.map`.

In [ ]:
%%time
# extract the train dataset
train_dataset = dataset['train']

# tokenize the train dataset is batches of 64
tokenised_dataset = train_dataset.map(tokenise_batch,
                                      batched=True, batch_size=64)
print(tokenised_dataset)

### Exercise: Tokenise all the sequences in the test split of the dataset

In [ ]:
# Get only the test section of the dataset
test_dataset = # Your code here
print(test_dataset)

In [ ]:
# Tokenise all the sequences in the column

In [ ]:
# Decode the sequences

#### Answer

In [ ]:
# Get only the test section of the dataset
test_dataset = dataset["test"]
print(test_dataset)

In [ ]:
%%time
# Tokenise all the sequences in the column
tokenised_dataset = test_dataset.map(tokenise_batch, batched=True, batch_size=64)
print(tokenised_dataset)

---

Finally, it is also possible to tokenize both sets at once:

In [ ]:
tokenised_dataset = dataset.map(tokenise_batch, batched=True, batch_size=64)


### Decode several sequences

In [ ]:
# Decode the sequences
tokens = list(tokenised_dataset['train']["input_ids"])[:10]
# Method 1: list`
decoded = tokeniser.batch_decode(tokens)
print(decoded)

In [ ]:
%%time
# Method 2: map
def decode_batch(batch):
  return {"tokens": tokeniser.decode(batch["input_ids"])}

decoded = tokenised_dataset.map(decode_batch, batched=True, batch_size=64)
print(decoded)
print(decoded['test']["tokens"])